# CNN for 52 Card + 1 Joker deck classification

In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import cv2
import matplotlib.pyplot as plt

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device is: {device}")

Device is: cuda


In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    #transforms.Resize((224, 224)),
])

train_dataset = datasets.ImageFolder(root='../card-dataset/train', transform=transform)
test_dataset = datasets.ImageFolder(root='../card-dataset/test', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Definición de la Red

In [5]:
import torch.nn as nn
import torch.nn.functional as F

CONV_LAYERS = [3, 32, 64]
FC_LAYERS = [CONV_LAYERS[-1] * 56 * 56, 120, 53]

# MODEL -------------------------------------------------------
class CardNet(nn.Module):
    def __init__(self):
        super(CardNet, self).__init__()
        self.conv1 = nn.Conv2d(CONV_LAYERS[0], CONV_LAYERS[1], kernel_size=3, stride=1, padding=2)

        self.pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)

        self.conv2 = nn.Conv2d(CONV_LAYERS[1], CONV_LAYERS[2], kernel_size=3, stride=1, padding=2)

        self.fc1 = nn.Linear(FC_LAYERS[0], FC_LAYERS[1])
        self.fc2 = nn.Linear(FC_LAYERS[1], FC_LAYERS[2])

    def forward(self, x):
        x = F.relu(self.conv1(x))
        #print(x.shape)
        x = self.pool(x)
        #print(x.shape)
        x = self.pool(F.relu(self.conv2(x)))
        #print(x.shape)
        x = x.view(-1, FC_LAYERS[0])
        #print(x.shape) # 1,
        x = F.relu(self.fc1(x))
        #print(x.shape)
        x = self.fc2(x)
        return x

 # Bucle de entrenamiento

In [ ]:
import torch.optim as optim

model = CardNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# TRAINING -------------------------------------------------------
EPOCHS = 10
for epoch in range(EPOCHS):
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {running_loss/len(train_loader)}')

Epoch 1, Loss: 3.814221799124235
Epoch 2, Loss: 2.508098091041693


## Bucle de evaluación

In [9]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Precisión del modelo en el conjunto de prueba: {100 * correct / total}%')

Precisión del modelo en el conjunto de prueba: 73.58490566037736%
